# Introduction to SpatialData analysis using 10x Genomics Xenium data

This tutorial explains basic [SpatialData](https://spatialdata.scverse.org/en/latest/index.html) operations and analysis:
* reading and saving files
* structure and contents of a SpatialData object
* visualisation of images, segmentations (shapes), gene expression, and annotations
* modifications: cropping, rotating and translating
* QC using scanpy
* Clustering
* Spatially variable genes


Additional and more in-depth tutorials are available at:
https://spatialdata.scverse.org/en/latest/tutorials/notebooks/notebooks.html

Information about the different functions of the SpatialData API is given here:
https://spatialdata.scverse.org/en/stable/api.html

Tutorials about analysis of spatial transcriptomics using scanpy and squidpy are available here (NOTE: not all of the are using SpatialData):
https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/index.html

## Dataset
In this tutorial, we are investigating the two samples of the 10x Genomics Alzheimer's disease mouse model Xenium dataset:

**Xenium In Situ Analysis of Alzheimer's Disease Mouse Model Brain Coronal Sections from One Hemisphere Over a Time Course** dataset, In Situ Gene Expression dataset analyzed using Xenium Onboard Analysis 1.4.0, 10x Genomics (CC BY 4.0 2023, July 13)

This dataset is available at: https://www.10xgenomics.com/datasets/xenium-in-situ-analysis-of-alzheimers-disease-mouse-model-brain-coronal-sections-from-one-hemisphere-over-a-time-course-1-standard
under the Creative Commons Attribution 4.0 International (CC BY 4.0) license.

We are using two samples derived from coronal mouse brain sections:

* age: 5.7 months
* sex: male
* one wildtype sample (WT)
* one disease model sample: transgenic CRND8 APP-overexpressing (TgCRND8) mouse

For more information on the TgCRND8 Alzheimer's mouse model and related models see:

* Chishti, Yang, Janus, Phinney, Horne, Pearson et al. (2001). Early-onset amyloid deposition and cognitive deficits in transgenic mice expressing a double mutant form of amyloid precursor protein 695. J. Biol. Chem.276, 21562–21570. [doi:10.1074/jbc.M100710200](https://www.sciencedirect.com/science/article/pii/S0021925820787296)
* Yokoyama, Kobayashi, Tatsumi, & Tomita  (2022). Mouse models of Alzheimer’s disease. Frontiers in Molecular Neuroscience, 15, 912995. [doi:10.3389/fnmol.2022.912995](https://www.frontiersin.org/journals/molecular-neuroscience/articles/10.3389/fnmol.2022.912995/full)

Gene panel:
* 247 probes from the [Xenium Mouse Brain Gene Expression Panel](https://www.10xgenomics.com/support/software/xenium-panel-designer/latest/tutorials/pre-designed-xenium-v1)
* 99 probes targeting activated microglia, astrocytes, and plaque-inducible genes

The data has already been downloaded. You can find the files in the following directories:

In [ ]:
path_xenium_disease = "/home/training/course_dir/data_dir/intro_spatialdata/datasets/xenium_10X_mouse_alzheimers/Xenium_V1_FFPE_TgCRND8_5_7_months_outs"
path_xenium_wt = "/home/training/course_dir/data_dir/intro_spatialdata/datasets/xenium_10X_mouse_alzheimers/Xenium_V1_FFPE_wildtype_5_7_months_outs"

Let us have a look at the wildtype sample:

In [ ]:
import os

os.listdir(path_xenium_wt)

These files are the typical output of the 10X Genomics Xenium Onboard Analysis Software (https://www.10xgenomics.com/support/software/xenium-onboard-analysis/latest/algorithms-overview/xoa-algorithms). Among others, this includes the following files related to:
* DAPI staining images: morphology_mip.ome.tif, morphology_focus.ome.tif
* Nuclei and cell segmentation: nucleus_boundaries.parquet, nucleus_boundaries.csv.gz, cell_boundaries.parquet, cell_boundaries.csv.gz
* Detected transcripts: transcripts.parquet, transcripts.csv.gz
* Cell-based count matrix: cell_feature_matrix.h5
* Summary and processing: analysis_summary.html, metrics_summary.csv

## Loading and saving files

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import spatialdata as sd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

Before we start looking at the data, we defining an output directory where we will store the initial SpatialData object: 

In [ ]:
outdir = "/home/training/analysis/spatialdata"
path_spatialdata_disease = outdir + "/TgCRND8_5_7_months.zarr"

path_spatialdata_wt = outdir + "/widltype_5_7_months.zarr"

Next, we are using the SpatialData reader for Xenium data. The readers are part of the 'spatialdata-io' package. Have look at following URL to see currently supported technologies: https://spatialdata.scverse.org/projects/io/en/latest/

We read the Xenium into SpatialData object `sdata_wt`:

In [ ]:
from spatialdata_io import xenium

sdata_wt = xenium(path_xenium_wt)

This will create a SpatialData object, which is stored in memory. We will write the object in Zarr format to the disk. Zarr is a very efficient format for working with large data sets on distributed systems such as cloud computing, for details see: https://zarr.dev/.

In [ ]:
sdata_wt.write(path_spatialdata_wt)

An advantage of the Zarr format is that we do not have to store the entire object in memory; but only the index to the files and parts that we are currently working with. This is in particular important for working with large spatial transcriptomics datasets. Currently, the entire SpatialData object is stored in memory. To make use of the memory-reducing Zarr functionality, we are reading the saved Zarr object and replace 'sdata_wt' with it:

In [ ]:
sdata_wt = sd.read_zarr(path_spatialdata_wt)

**A note on reading and saving SpatialData objects**: to avoid collisions, you cannot overwrite a SpatialData object while you are working with it. 
This means that you should always save a SpatialData object to a path that differs from the one that you used to read it from. 
You can, however, overwrite an existing SpatialData object if you did not load it in memory by specifying the `overwrite=True` flag.

The SpatialData objects contains the following entries:

In [ ]:
sdata_wt

* Images: two versions of the DAPI image, a focus ('morphology_focus') and maximum intensity projection ('morphology_mip') in five different resolutions
* Labels: based on initial clustering NOTE: we can ignore the labels as we will re-compute clusters in steps below.
* Points: the actual coordinates of detected 'transcripts'
* Shapes: segmentations as polygons of the nuclei ('nucleus_boundaries'), cells ('cell_boundaries') and as circles for cells ('cell_circles')
* 'Tables': an AnnData object that contains the cell-based count matrix and meta information for cells and transcripts
* 'coordinate systems': this relates to the common coordinates between the images and shapes. In this tutorial, we will only be working with the single 'global' coordinate system

We also create a zarr object and load it for the disease sample.

In [ ]:
from spatialdata_io import xenium

sdata_disease = xenium(path_xenium_disease)
sdata_disease.write(path_spatialdata_disease)
sdata_disease = sd.read_zarr(path_spatialdata_disease)

Let us look at the 'table' AnnData object now. For a description of AnnData, see here: [https://anndata.readthedocs.io/en/latest/](https://anndata.readthedocs.io/en/latest/)

In [ ]:
sdata_wt['table']

This object contains entries for 58,685 segmented cells ('n_obs') and 347 probes, i.e. genes in this case ('n_vars').

* 'obs': a data frame with meta information and metrics for each cell
* 'var': a data frmae that contains information for each gene
* 'uns': contains additional information, which does not fit into the 'obs' or 'var' data frame format
* 'obsm': matrices related to the cells. For example, 'spatial' comprises the spatial coordinates of each cell

The count matrix in the form of a sparse matrix is inside the 'X' entry.

In [ ]:
sdata_wt['table'].X

The 'var' data frame contains the gene names as an index and ENSEMBL gene IDs:

In [ ]:
sdata_wt['table'].var

The 'obs' data frame comprises entries such as the cell IDs, metrics including the number of transcripts per cell ('transcript_counts') and Xenium quality score related entries ('control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts' and 'deprecated_codeword_counts').

In [ ]:
sdata_wt['table'].obs

As additional metadata, we are specifying the name of the `sample` and the `condition`.

In [ ]:
sdata_wt['table'].obs['condition'] = 'wildtype'
sdata_wt['table'].obs['sample'] = 'V1_FFPE_wildtype_5_7_months'
sdata_wt['table'].obs

In [ ]:
sdata_disease

In [ ]:
sdata_disease['table'].obs['condition'] = 'TgCRND8'
sdata_disease['table'].obs['sample'] = 'V1_FFPE_TgCRND8_5_7_months'
sdata_disease['table'].obs

## Images and shapes

To visualise images, shapes, and data, we are using the 'spatialdata-plot' package. This package has to be loaded for plotting functions to work properly.

In [ ]:
import spatialdata_plot

Properties of images can be inspected with the `get_pyramid_levels` command. 'n' will select the different the resolutions of the image.

In [ ]:
sd.get_pyramid_levels(sdata_wt["morphology_mip"], n=0)

This shows us that 'morphology_focus' is a 2D image with a single channel ('c') with a width of 45,409 and height of 23,866. The 'transform' entry 'Identity' means that no transformation has been applied to this image.

`get_channel_names` can be used to retrieve the names of channels. In this case, there is a single channel called 0.

In [ ]:
sd.models.get_channel_names(sdata_wt["morphology_mip"])

Visualising images in SpatialData is done by following steps:
1) you specify which image to render and the color map ('cmap') that should be used: `pl.render_images(..)`
2) you create the plot: `pl.show(..)`

In [ ]:
sdata_wt.pl.render_images("morphology_mip", cmap="gray").pl.show(title=f"Morphology image (MIP) - wildtype", coordinate_systems="global", figsize=(10, 5))

We can see two distinct pieces of tissue in this plot.

Instead of plotting a gray image on black background you can specify another color map like "Purples"'. You can find available colormaps here: https://matplotlib.org/stable/gallery/color/colormap_reference.html

In [ ]:
sdata_wt.pl.render_images("morphology_mip", cmap="Purples").pl.show(title=f"Morphology image (MIP) - wildtype", coordinate_systems="global", figsize=(10, 5))

In [ ]:
sdata_disease.pl.render_images("morphology_mip", cmap="Purples").pl.show(title=f"Morphology image (focus) - disease", coordinate_systems="global", figsize=(10, 5))

To render segmentations (shapes) instead of the DAPI images, you can use the `pl.render_shapes` command.

In [ ]:
sdata_wt.pl.render_shapes("cell_circles", color="darkblue").pl.show(title=f"Cell cirles - wildtype",)

## Transformations

Transformations are core features of SpatialData. These transformations include:

* Crop a SpatialData object. This will subset the object.
* Scale images and shapes
* Rotate images and shapes
* Translate images and shapes

Scale, rotation, and translation operations are applied to a specific instance (e.g. the morphology image) and coordinate system. Instead of directly modifying coordinates of the input images and shapes, SpatialData objects save the active transformations and apply them.

#### Crop data 

The `bounding_box_query` is a handy command to crop the SpatialData object. This is useful to zoom into specific regions of the data.

To zoom in, we use a bounding box with width and height of 5000 with an offset of 5000. We store the cropped object in `crop_sdata_wt`:

In [ ]:
crop_sdata_wt = sd.bounding_box_query(
    sdata_wt, axes=("x", "y"), min_coordinate=[5000, 5000], max_coordinate=[10000, 10000], target_coordinate_system="global"
)
crop_sdata_wt

As you can see, the 'bounding_box_query' did not only crop the images and shapes but also reduced the AnnData object to a smaller number of cells (3,342 cells).

For example, we can inspect the nuclei segmentation this way in more detail.

In [ ]:
crop_sdata_wt.pl.render_images("morphology_mip", cmap="Purples").pl.render_shapes("nucleus_boundaries", color="red").pl.show(title="Segmeneted nuclei over morphology - wildtype")

And how the cell circle segmentation mask relates to it by plotting nuclei ontop of the cell circle segmentation mask. 

In [ ]:
crop_sdata_wt.pl.render_shapes("cell_circles", color="darkblue").pl.render_shapes("nucleus_boundaries", color="red").pl.show(title="Cell circles and nuclei - wildtype")

If we are not using it we can use `del` to delete it and free the memory associated with this object:

In [ ]:
del crop_sdata_wt

### Rotate and translate images and shapes

SpatialData operations can be used to rotate and translate images and shapes.

For convenience, we create a function `get_xy_rotation_by_degrees` that takes degrees as input and returns a SpatialData rotation.

In [ ]:
import math
import spatialdata as sd

def get_xy_rotation_by_degrees(degrees: float, small_digits: int=15) -> sd.transformations.transformations.Affine: 
    """
    Generates a SpatialData rotation on x-axis and y-axis using degrees as input.

    Parameters
    ----------
    degrees: float
        The number degrees images and shapes should be rotated
    small_digits: int 
        The number of digits to consider for sine and cosine computation to avoid numerical noise

    Returns
    -------
    sd.transformations.transformations.Affine
        SpatialData rotation that can be applied to SpatialData instances (images and shapes)
    """
    
    rad = math.radians(degrees)

    rotation = sd.transformations.Affine(
        [
            [round(math.cos(rad), small_digits), -round(math.sin(rad), small_digits), 0],
            [round(math.sin(rad), small_digits), round(math.cos(rad), small_digits), 0],
            [0, 0, 1],
        ],
        input_axes=("x", "y"),
        output_axes=("x", "y"),
    )

    return rotation

SpatialData rotations are defined as a matrix. For instance, for a 90 degrees rotation our function generates the following rotation matrix:

In [ ]:
get_xy_rotation_by_degrees(90)

Before we apply this transformation, we should check whether other transformations have already been applied to the image or segmentation (shape). We would overwrite existing transformation otherwise. Existing transformations can be checked with `transformations.get_transformation` command.

In [ ]:
sd.transformations.get_transformation(sdata_wt.images["morphology_mip"])

'Identity' means no transformations has been applied to morphology image so far.

Let us first apply a 90 degrees rotation to the morphology focus image. This is done by using the `set_transformation` command.

In [ ]:
rotation = get_xy_rotation_by_degrees(90)

sd.transformations.set_transformation(sdata_wt.images["morphology_mip"], rotation, to_coordinate_system="global")

sdata_wt.pl.render_images().pl.show()

After rendering all images, labels, and shapes, we can see that the morphology image has been rotated but not the other images, labels, and shapes.

In [ ]:
sdata_wt.pl.render_images("morphology_mip", cmap="Purples").pl.show(title=f"Morphology image (MIP) - wildtype", coordinate_systems="global", figsize=(10, 5))

The morphology image exhibits some negative x-coordinates due to the rotation. We can get the maximum and minimum x- and y-coordinates with the function `get_extent`:

In [ ]:
sd.get_extent( sdata_wt.images['morphology_mip'] )

To shift everything into non-negative coordinates, we can apply a translation to the x-axis based on the minimum x-coordinate using the `transformations.Translation` function:

In [ ]:
translation = sd.transformations.Translation([24869, 0], axes=("x", "y"))

To rotate and then translate the morphology image, we have to specify a `transformations.Sequence`:

In [ ]:
transf_sequence = sd.transformations.Sequence([rotation, translation])
transf_sequence

In [ ]:
sd.transformations.set_transformation(sdata_wt.images["morphology_mip"], transf_sequence, to_coordinate_system="global")

In [ ]:
sd.transformations.get_transformation(sdata_wt.images["morphology_mip"])

In [ ]:
sdata_wt.pl.render_images("morphology_mip", cmap="Purples").pl.show(title=f"Morphology image (focus - wildtype", coordinate_systems="global", figsize=(10, 5))

However, the cell circles have already been scaled. We have to make sure that we scale them when specifying new transformations.

In [ ]:
sd.transformations.get_transformation(sdata_wt.shapes["cell_circles"])

For the cell circles, we have to include the already existing scale transformation in our sequence of transformations.

In [ ]:
existing_transformations = sd.transformations.get_transformation(sdata_wt.shapes["cell_circles"])

In [ ]:
existing_transformations

In [ ]:
transf_sequence = sd.transformations.Sequence([existing_transformations, rotation, translation])
transf_sequence

In [ ]:
sd.transformations.set_transformation(sdata_wt.shapes["cell_circles"], transf_sequence, to_coordinate_system="global")

In [ ]:
sdata_wt.pl.render_shapes("cell_circles", color="darkblue").pl.show()

### TASK: rotate and translate disease sample

It should be rotated in a way that that it would form the right brain hemisphere given the wildtype sample is the left brain hemisphere.

**Hint:** follow the same steps as above for the morphologi (MIP) image an cell circle segmentation mask
1) always check for existing transformations
2) define the rotation with the desired number of degrees
3) check the extent of the data points
4) define translation
5) create the sequence of transformations with the correct and apply it

In case you did not manage to transform the data properly, you can load the completely processed object like this:

In [ ]:
# uncomment below 
# path_final_wildtype = "/home/training/course_dir/data_dir/intro_spatialdata/datasets/xenium_10X_mouse_alzheimers/finalized_wildtype_5_7_months.zarr"
# path_final_disease = "/home/training/course_dir/data_dir/intro_spatialdata/datasets/xenium_10X_mouse_alzheimers/finalized_TgCRND8_5_7_months.zarr"

# sdata_wt = sd.read_zarr(path_final_wildtype)
# sdata_disease = sd.read_zarr(path_final_disease)

# reset counts which will be modified below
# sdata_wt['table'].X = sdata_wt['table'].layers['counts'].copy()
# sdata_disease['table'].X = sdata_disease['table'].layers['counts'].copy()

## View gene expression

To view the expression of a gene, you can simply specify that the gene in the 'color' parameter of the `pl.render_shapes`. For example, here we show the expression of the upper layer necortex neuron marker gene 'Cux2'.

In [ ]:
gene_name = "Cux2"
sdata_wt.pl.render_shapes(
    "cell_circles",
    color=gene_name,
    cmap="Reds"
).pl.show(title=f"{gene_name} expression - wildtype", coordinate_systems="global", figsize=(10, 5))

In [ ]:
gene_name = "Cux2"
sdata_disease.pl.render_shapes(
    "cell_circles",
    color=gene_name,
    cmap="Reds"
).pl.show(title=f"{gene_name} expression - disease", coordinate_systems="global", figsize=(10, 5))

### Quality control

SpatialData seamlessly integrates with [scanpy](https://scanpy.readthedocs.io/en/stable/) and [squidpy](https://squidpy.readthedocs.io/en/stable/). You can simply use scanpy and squidpy functions and apply them to the 'table' AnnData object inside the SpatialData objects.

For the QC and clustering the samples together, it can be useful to combine the anndata objects. During the processing we will combine and store them outside of the SpatialData object.

Anndata objects can be combined using the `concatenate` function. This function will append a batch number to the cell indexes seperated by `___` in case we have matching cell indexes between samples.

In [ ]:
adata_combined = sdata_wt['table'].concatenate(sdata_disease['table'], index_unique = "___", uns_merge = "unique")
adata_combined

In [ ]:
adata_combined.obs

After combining the samples, we can use the scanpy command `pp.calculate_qc_metrics` to compute QC metrics.

In [ ]:
sc.pp.calculate_qc_metrics(adata_combined, percent_top=None, log1p=False, inplace=True)

In [ ]:
adata_combined.obs

This adds a column for the number of genes per cell ('n_genes_by_counts') and the total number of detected transcripts ('total_counts').

To obtain an idea about the different levels of segmentation, we can compute the ratio between the area of the segmented nucleus and cell for each cell.

In [ ]:
adata_combined.obs["nucleus_ratio"] = adata_combined.obs["nucleus_area"] / adata_combined.obs["cell_area"]

Violin plots can be used to investigate the distribution of these QC metrics all the cells.

In [ ]:
sc.pl.violin(
    adata_combined,
    ["n_genes_by_counts", "total_counts"],
    jitter=False,
    multi_panel=True,
    groupby='sample'
)

In [ ]:
sc.pl.violin(
    adata_combined,
    ["cell_area", "nucleus_ratio"],
    jitter=False,
    multi_panel=True,
    groupby='sample'
)

In [ ]:
adata_combined

To quickly look at the QC metrics on spatial coordinates, we can subset the anndata object by condition and use the squidpy `sq.pl.spatial_scatter` function to plot them.

NOTE: Given that SpatialData is applying transformations on the fly without altering the real coordinates, the data is not rotated when using squidpy for plotting

In [ ]:
import squidpy as sq

sq.pl.spatial_scatter(adata_combined[ adata_combined.obs['condition'] == 'wildtype', :], color=['n_genes_by_counts', 'total_counts', 'cell_area'], groups='condition', shape=None)

In [ ]:
import squidpy as sq

sq.pl.spatial_scatter(adata_combined[ adata_combined.obs['condition'] == 'TgCRND8', :], color=['n_genes_by_counts', 'total_counts', 'cell_area'], groups='condition', shape=None)

### Basic processing

Processing of spatial transcriptomics data is very similar to processing single cell data:

* Data normalisation
* PCA
* Clustering
* Cell type annotation (out of scope for this tutorial)
* Spatial transcriptomics data specific analysis such as identifying spatially variable genes

Since we will modify transcript counts per cell by normalising them, we store the raw counts into a layer called 'counts' in the AnnData object.

In [ ]:
# saw raw counts
adata_combined.layers['counts'] = adata_combined.X.copy()

A simple way of normalisation, which usually works well, is to normalize all the counts per cell using the scanpy function `pp.normalize_total`. To deal with heteroscedasticy in the data, we usually log-normalise the data with the `pp.log1p` command. The scanpy framework also offers more sophsiticated functions for normalisation such as using [Pearson residuals](https://scanpy.readthedocs.io/en/stable/generated/scanpy.experimental.pp.normalize_pearson_residuals.html). 

In [ ]:
sc.pp.normalize_total(adata_combined)
sc.pp.log1p(adata_combined)

### PCA

Principal component analysis (PCA) is a good start to identify genes that explain most of the variation in gene expression in a data set. The principal components will also be used downstream for clustering the cells. PCA often works best if it is limited to highly variable genes in the data set. We can detect highly variable genes with the scanpy command `pp.highly_variable_genes`.

In [ ]:
sc.pp.highly_variable_genes(adata_combined)

Among other columns it as flag 'highly_variable' in the 'var' dataframe.

In [ ]:
adata_combined.var.sort_values("means", ascending=False)

The actual PCA is performed with the scanpy function `tl.pca`. Here, we specify that is limited to highly variably genes: 'use_highly_variable=True'

In [ ]:
sc.tl.pca(adata_combined, use_highly_variable=True)

Elbow plots show which PCs explain most of the data. We use the scanpy function `pl.pca_variance_ratio` to plot them.

In [ ]:
sc.pl.pca_variance_ratio(adata_combined, n_pcs=50, log=True)

By plotting principal component we can investigate the influence of QC metrics and the different conditions on the variance in our data using the scanpy function `sc.pl.pca`: 

In [ ]:
sc.pl.pca(
    adata_combined, color=['total_counts', 'condition']
)

A first idea about interesting genes in our dataset can be obtained by looking at the loadings of the PCs.

In [ ]:
sc.pl.pca_loadings(adata_combined, include_lowest=False)

In [ ]:
sc.pl.pca(
    adata_combined, color=['Plp1', 'Mbp', 'Gad1']
)

### Clustering

While PCs can provide interesting insights, UMAPs are often more usual to visualise how cells cluster together based on their expression profiles. UMAPs are computed based on a neighborhood graph that describes the similarities between cells. The neighborhood graph is computed with the scanpy function `pp.neighbors` and uses the PCs as input. UMAPs can then be computed the 'tl.umap' function.

In [ ]:
sc.pp.neighbors(adata_combined)
sc.tl.umap(adata_combined)

`pl.umap` is the command for plotting UMAPs. 

In [ ]:
sc.pl.umap(adata_combined, color=['total_counts', 'condition'])

Clusters are inferred from the neighborhood graph using Leiden clustering. The scanpy function `tl.leiden` takes the AnnData object as input. The clustering resolution can be specified by the 'resolution' parameter. It is advisable to try several different clustering resolution and inspect the UMAP, this can be done by looping over the resolutions: 

In [ ]:
clus_resolutions = [0.25, 0.5, 1.0, 1.5, 2.0]

for resolution in clus_resolutions:
    print(f"Clustering resolution {resolution} ...")
    sc.tl.leiden(adata_combined, resolution=resolution, key_added='leiden_' + str(resolution), flavor='igraph', n_iterations=2)

Cluster assignments are added in the `obs`table. Multiple clusterings with different resolutions can be added by explicitly specifying the name for the clustering as the `key_added` parameter as we did above.

In [ ]:
sc.pl.umap(
    adata_combined,
    color=[
        "total_counts",
        "n_genes_by_counts",
        "leiden_0.25", "leiden_0.5", "leiden_1.0", "leiden_1.5", "leiden_2.0"
    ],
    wspace=0.4,
    ncols= 2
)

For this tutorial, we are picking a resolution of 0.5 which gives us a sufficient seperation of cell clusters. Selecting a clustering resolution is arbitrary and depends on your research question, for example, if you are interested in investigating broad cell type categories, subtypes or supertypes.

In [ ]:
clustering = "leiden_0.5"

sc.pl.umap(
    adata_combined,
    color=[
    clustering, 
    ],
    wspace=0.4,
    legend_loc = 'on data'
)

Note that the clusterings are solely computed based on gene expression profiles of the cells with out considering any spatial information of the cells. 
However, these clusters often reflect cell types or states with distinct spatial locations as you will see when we plot them on the spatial coordinates below.

### Updating the SpatialData object and plotting clusters on spatial coordinates
For updating the anndata table in the SpatialData objects, we have to:
1) split the combined anndata object into two different object by condition again
2) and then remove the batch identifier from the cell indexes
3) substitute the anndata table in the SpatialData objects

In [ ]:
adata_wt = adata_combined[adata_combined.obs['condition'] == 'wildtype'].copy()
adata_disease = adata_combined[adata_combined.obs['condition'] == 'TgCRND8'].copy()

We can simple split the cell indexes by the separator `___` to restore the original indexes

In [ ]:
adata_wt.obs_names = [idx.split('___')[0] for idx in  adata_wt.obs_names ]
adata_wt.obs

In [ ]:
adata_disease.obs_names = [idx.split('___')[0] for idx in  adata_disease.obs_names ]
adata_disease.obs

When copying them to the SpatialData object using `.copy()` ensures that we do not just create a view that is pointing to the anndata object.

In [ ]:
sdata_wt['table'] = adata_wt.copy()
sdata_disease['table'] = adata_disease.copy()

After updating the SpatialData table anndata, we can use the SpatialData plot functions to visualize clusters and other entries that we added to the anndata object.

In [ ]:
clustering = 'leiden_0.5'

sdata_wt.pl.render_shapes(
    "cell_circles",
    color=clustering,
    cmap="Reds"
).pl.show(title=f"{clustering} - wildtype", coordinate_systems="global", figsize=(10, 5))

In [ ]:
clustering = 'leiden_0.5'

sdata_disease.pl.render_shapes(
    "cell_circles",
    color=clustering,
    cmap="Reds"
).pl.show(title=f"{clustering} - disease", coordinate_systems="global", figsize=(10, 5))

## Marker genes

To obtain an understanding about the characteristics of cells it is useful to compute marker genes. This can be done with the scanpy function `sc.tl.rank_genes_groups`. Per default it will compare all cells of one cluster against all other cells. We are using a Wilcoxon rank-sum test. The `pts` parameter computes the number of cells with non-zero expression of a gene in addition. A detailed description is given in the [scanpy.tl.rank_genes_groups](https://scanpy.readthedocs.io/en/stable/generated/scanpy.tl.rank_genes_groups.html) Read the Docs.

In [ ]:
sc.tl.rank_genes_groups(adata_combined, groupby=clustering, method='wilcoxon', pts=True)

Dot plots are a good way to summarise marker genes per cluster using the `sc.pl.rank_genes_groups_dotplot` function.

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_combined, groupby=clustering, standard_scale="var", n_genes=3)

## Cell type annotation

Gene panels like the Xenium brain panel are specifically designed to include strong marker genes. Using these genes to annotate cells, usually requires prior knowledge. For simplicity, we specify marker gene signatures for broad cell type categories in this tutorial:

* Microglia
* Astrocytes
* Glutamatergic neurons
* GABAergic neurons

In [ ]:
microglia_markers = ['C1qa', 'C1qb', 'Csf1r', 'Spi1', 'Tmem119']
astrocyte_markers = ['Aqp4', 'Rorb', 'Gfap']
oligo_markers = ['Sox10', 'Plp1', 'Pdgfra', 'Olig2']
glut_markers = ['Slc17a6', 'Slc17a7']
gaba_markers = ['Gad1', 'Gad2']

Simply by investigating the expression of these genes, we can understand what kinds of cells the different clusters correspond to.

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 5, figsize=[15, 5])

sc.pl.dotplot(adata_combined, groupby=clustering, standard_scale="var", var_names=microglia_markers, ax=axs[0], return_fig=False, show=False)
axs[0].set_title('Microglia')
sc.pl.dotplot(adata_combined, groupby=clustering, standard_scale="var", var_names=astrocyte_markers, ax=axs[1], return_fig=False, show=False)
axs[1].set_title('Astrocyte')
sc.pl.dotplot(adata_combined, groupby=clustering, standard_scale="var", var_names=oligo_markers, ax=axs[2], return_fig=False, show=False)
axs[2].set_title('Oligodendrocyte')
sc.pl.dotplot(adata_combined, groupby=clustering, standard_scale="var", var_names=glut_markers, ax=axs[3], return_fig=False, show=False)
axs[3].set_title('Glutamatergic neuron')
sc.pl.dotplot(adata_combined, groupby=clustering, standard_scale="var", var_names=gaba_markers, ax=axs[4], return_fig=False, show=False)
axs[4].set_title('GABAergic neuron')

### Differential expression
The `sc.tl.rank_genes_groups` function be used to investigate differential expression, too. 
To investigate differentially expressed genes between wildtype and disease per cluster:
1) we specify in addition meta data entry, where we split cells by clusters and by condition (wildtype vs. disease)
2) compute differential expression per cluster


In [ ]:
# compare expression per cluster
adata_combined.obs['condition_' + clustering] = [ condition + "_" + cluster for condition, cluster in zip(adata_combined.obs['condition'], adata_combined.obs[clustering]) ]

In [ ]:
import numpy as np
conditions = np.unique(adata_combined.obs['condition'])
conditions

To use the `sc.tl.rank_genes_groups` for differential expression we have to explictly specify the group for the contrast:
* `groupby`: specifies the group column to consider
* `groups`: has to be specified in addition, otherwise the function would still use all cells as a background
* `reference`: the group we compare the group against: wildtype as reference for up-regulated genes in disease
  
We add the results of each contrast as `contrast_condition__' + clustering + '__' + clus` in the `uns` entry of the anndata object.

In [ ]:
conditions = np.unique(adata_combined.obs['condition'])

for clus in np.unique(adata_combined.obs[clustering]):
    group1 = 'TgCRND8' + '_' + clus
    group2 = 'wildtype' + '_' + clus

    if adata_combined[adata_combined.obs['condition_' + clustering] == group1 ].shape[0] > 1 \
        and adata_combined[adata_combined.obs['condition_' + clustering] == group2 ].shape[0] > 1:
        contrast = 'contrast_condition__' + clustering + '__' + clus
        sc.tl.rank_genes_groups(adata_combined, groupby='condition_' + clustering, groups=[group1, group2], 
                                reference=group2, key_added = contrast, pts=True)

In [ ]:
adata_combined

We we can look up the differentially expressed genes per cluster between disease and wildtype. We filter them by p-value and the percentage of cells that express the gene `pct_nz_group`.
For example for cluster 0:

In [ ]:
cluster = '0'
contrast = 'contrast_condition__leiden_0.5__' + cluster
group1 = 'TgCRND8' + '_' + cluster

diff_genes = sc.get.rank_genes_groups_df(adata_combined, key=contrast, group=group1).sort_values('logfoldchanges', ascending=False)
diff_genes[ (diff_genes.pvals_adj < 0.05 ) &  (diff_genes.pct_nz_group > 0.1) ]

We can visualize these expression of these genes in dot plots using the `sc.pl.dotplot` function:

In [ ]:
selected_genes = ['Penk', 'Osmr', 'Ccn2', 'Arc', 'S100a6']

sc.pl.dotplot(adata_combined[ adata_combined.obs[clustering] == cluster] , 
              groupby='condition_' + clustering, standard_scale="var", var_names=selected_genes)

And plot them on spatial coordinates:

In [ ]:
gene_name = "Penk"
sdata_wt.pl.render_shapes(
    "cell_circles",
    color=gene_name,
    cmap="Reds"
).pl.show(title=f"{gene_name} expression - wildtype", coordinate_systems="global", figsize=(10, 5))

### TASK: identify up-regulated genes in microglia between disease and wiltype
1) identify differentially expressed genes
2) visualize them in dot plots
3) visualize their expression on spatial coordinates
4) zoom-in to the hippocampal area and look at the expression of one of the genes for the wildtype and disease sample

## Spatially variable genes

To make use of spatial properties of our data, we can use the [squidpy](https://squidpy.readthedocs.io/en/stable/) package. As an example, we will identify spatially variable genes. Squidpy contains a function for computing a spatial neighborhood graph 'gr.spatial_neighbors'.

In [ ]:
adata_wt

To identify differentially expressed genes per condition, we split the combined anndata object into two separate ones.

In [ ]:
adata_wt = adata_combined[adata_combined.obs['condition'] == 'wildtype'].copy()
adata_disease = adata_combined[adata_combined.obs['condition'] == 'TgCRND8'].copy()

In [ ]:
import squidpy as sq

sq.gr.spatial_neighbors(adata_wt, coord_type="generic", delaunay=True)
sq.gr.spatial_neighbors(adata_disease, coord_type="generic", delaunay=True)

Spatially variable genes can be detected based on their spatial autocorrelation. An old but established statistic to detect spatial autocorrelation is [Moran's I](https://en.wikipedia.org/wiki/Moran%27s_I). Moran's I for all genes can be calculated with the `gr.spatial_autocorr` function that utlizes the spatial neighborhood graph.

In [ ]:
import squidpy as sq

sq.gr.spatial_neighbors(adata_wt, coord_type="generic", delaunay=True)
sq.gr.spatial_autocorr(
    adata_wt,
    mode="moran",
    n_perms=100,
    n_jobs=8,
)

In [ ]:
import squidpy as sq

sq.gr.spatial_neighbors(adata_disease, coord_type="generic", delaunay=True)
sq.gr.spatial_autocorr(
    adata_disease,
    mode="moran",
    n_perms=100,
    n_jobs=8,
)

A ranked list of specially variable genes is stored inside the 'uns' entry 'moranI'.

In [ ]:
adata_wt.uns["moranI"].head(10)

In [ ]:
adata_disease.uns["moranI"].head(10)

The two most spatially variable genes exhibit characteristic expression pattern across the brain section that we are looking at.

In [ ]:
gene_name = "Mbp"
sdata_wt.pl.render_shapes(
    "cell_circles",
    color=gene_name,
    cmap="Reds"
).pl.show(title=f"{gene_name} expression - wildtype", coordinate_systems="global", figsize=(10, 5))

In [ ]:
gene_name = "Slc17a7"
sdata_wt.pl.render_shapes(
    "cell_circles",
    color=gene_name,
    cmap="Reds"
).pl.show(title=f"{gene_name} expression - wildtype", coordinate_systems="global", figsize=(10, 5))

### Save spatialdata objects

To complete this tutorial, we save the processed spatialData objects in a new Zarr folder.

In [ ]:
sdata_wt['table'] = adata_wt.copy()
sdata_disease['table'] = adata_disease.copy()

In [ ]:
outdir = "/home/training/analysis/spatialdata"
path_spatialdata_wt = outdir + "/finalized_wildtype_5_7_months.zarr"

outdir = "/home/training/analysis/spatialdata"
path_spatialdata_disease = outdir + "/finalized_TgCRND8_5_7_months.zarr"


sdata_wt.write(path_spatialdata_wt, overwrite=True)
sdata_disease.write(path_spatialdata_disease, overwrite=True)